# YOLOv8s — Test-set evaluation

Оценка чекпоинта `weak` (из augmentation_study) на `test`-сплите BrackishMOT.
Параметры валидации совпадают с `train_yolo26s.ipynb` — честное сравнение архитектур.

In [1]:
import os
from pathlib import Path
from ultralytics import YOLO

if Path.cwd().name == "notebooks":
    os.chdir("..")
print(f"Working dir: {Path.cwd()}")

import torch
print(f"PyTorch:     {torch.__version__}")
print(f"CUDA:        {torch.cuda.is_available()} — {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

Working dir: d:\Slermo\Programming\Python\yolo_marine
PyTorch:     2.10.0+cu128
CUDA:        True — NVIDIA GeForce RTX 5060 Ti


## Поиск чекпоинта

ultralytics автоинкрементит имя рана (`weak`, `weak2`, ...) — берём самый свежий weak-ран по mtime.

In [2]:
candidates = sorted(
    Path("runs/detect/runs/augmentation_study").glob("weak*/weights/best.pt"),
    key=lambda p: p.stat().st_mtime,
)
if not candidates:
    raise FileNotFoundError(
        "Не найден вес v8s в runs/detect/runs/augmentation_study/weak*/weights/best.pt. "
        "Укажи путь вручную."
    )
WEIGHTS = candidates[-1]
print(f"Using: {WEIGHTS}")

Using: runs\detect\runs\augmentation_study\weak\weights\best.pt


## Evaluate on test set

In [3]:
best = YOLO(str(WEIGHTS))

metrics = best.val(
    data="configs/dataset.yaml",
    split="test",
    batch=1,
    device=0,
    plots=True,
)

print(f"mAP@50:      {metrics.box.map50:.4f}")
print(f"mAP@50-95:   {metrics.box.map:.4f}")
print(f"Precision:   {metrics.box.mp:.4f}")
print(f"Recall:      {metrics.box.mr:.4f}")

print(f"\nInference speed: {metrics.speed}")

print("\nPer-class AP@50:")
for i, name in metrics.names.items():
    print(f"  {name:12s}  {metrics.box.ap50[i]:.4f}")

Ultralytics 8.4.19  Python-3.12.3 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5060 Ti, 16310MiB)
Model summary (fused): 73 layers, 11,127,906 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 1055.9204.3 MB/s, size: 332.9 KB)
val: Scanning D:\Slermo\Programming\Python\yolo_marine\data\brackish-dataset\labels\test.cache... 1468 images, 229 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1468/1468  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1468/1468 69.5it/s 21.1s<0.1s
                   all       1468       3466       0.99      0.958      0.984      0.844
                  fish        321        321       0.99      0.969      0.987      0.872
            small_fish        247        965      0.976      0.919      0.973      0.742
                  crab        614       1279      0.996       0.99      0.995      0.956
                shrimp         57         57      0.982      0.938    